In [0]:
from pyspark.sql.functions import col, when, count, avg, sum, round

orders = spark.read.table('workspace.ecommerce_gold.orders')
order_items = spark.read.table('workspace.ecommerce_silver.order_items')
products = spark.read.table('workspace.ecommerce_gold.products')

product_features = order_items.groupBy('product_id').agg(
    count('order_item_id').alias('total_quantity_sold'),
    count('order_id').alias('total_orders'),
    sum(round(col('price'))).alias('total_revenue'),
    avg(round(col('price'))).alias('avg_price')
)

product_features = product_features.join(
    products.select('product_id', 'product_category_name', 'avg_review_score'), 
    'product_id', 'left'
)
display(product_features)

In [0]:
threshold = product_features.approxQuantile(
    'total_quantity_sold', [0.8], 0.1
)[0]

product_demand = product_features.withColumn(
    'high_demand_product',
    when(col('total_quantity_sold') >= threshold, 1).otherwise(0)
)

product_demand.groupBy('high_demand_product').count().show()

In [0]:
product_demand.select("high_demand_product").describe().show()

In [0]:
product_demand =product_demand.filter(
    col('product_category_name').isNotNull()
)

In [0]:
product_demand = product_demand.dropna(subset= ['avg_review_score'])

In [0]:
product_demand.orderBy(col('total_quantity_sold').desc()).select(
    'product_id', 'total_quantity_sold', 'product_category_name',
).show(20, truncate=False)

In [0]:
product_demand.groupBy('product_category_name').agg(
    sum('total_quantity_sold').alias('category_sold'),
    count('product_id').alias('num_products')
).orderBy(col('category_sold').desc()).show(20, truncate=False)


In [0]:
category_demand = product_demand.groupBy('product_category_name').agg(
    sum('total_quantity_sold').alias('cat_total_sold'),
    sum('total_orders').alias('category_total_orders'),
    sum('total_revenue').alias('category_total_revenue'),
    count('product_id').alias('num_products')
)

category_demand.orderBy(
    col('cat_total_sold').desc()
).show(20, truncate= False)


In [0]:
category_high_demand = product_demand.groupBy('product_category_name').agg(
    sum('total_quantity_sold').alias('category_total_sold'),
    sum('total_revenue').alias('total_revenue'),
    avg('high_demand_product').alias('high_demand_rate'),
    count('product_id').alias('num_products')
)

category_high_demand.filter(
    col('num_products') >= 50
).orderBy(
    col('category_total_sold').desc()
).show(20, truncate=False)


In [0]:
category_high_demand.write.format('delta') \
.mode('overwrite') \
.option('overwriteSchema', 'true') \
.saveAsTable('workspace_ecommerce_category_high_demand_analysis')        

In [0]:
product_demand.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace_ecommerce_product_demand')    